# 실습1 · 데이터 적재와 임베딩

KLUE-MRC 지문·질문을 PostgreSQL 에 넣고, LM Studio(Qwen3-Embedding-8B)로 임베딩을 채운다.

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 — 주소·모델·부하를 이 셀에서만 고친다 |
| 2 | 라이브러리와 DB 연결 |
| 3 | 데이터셋 받기 (볼륨에 없을 때만) |
| 4 | 지문·질문 표로 정리 |
| 5 | 테이블 비우고 텍스트 적재 |
| 6 | 임베딩 서버 확인 |
| 7 | 배치 임베딩 함수 |
| 8 | 지문 임베딩 |
| 9 | 질문 임베딩 |
| 10 | 인덱스 만들기와 확인 |
| 11 | 맛보기 검색 |

- **실행** : 셀 1을 확인한 뒤 메뉴 **Run → Run All Cells**
- **멈춤** : **Kernel → Interrupt Kernel** (배치마다 저장하므로 채운 행은 남는다)
- **이어서** : **셀 6** 을 클릭하고 **Run → Run Selected Cell and All Below** (Run All Cells 는 셀 5에서 테이블을 다시 비운다)
- **진행 확인** : 탭을 새로 고치면 진행 출력이 멈춘 것처럼 보여도 계산은 계속된다 → `01b_진행확인.ipynb` 로 확인

### 셀 1 · 설정
이 노트북에서 고치는 값은 여기에만 있다.
- `LMSTUDIO_URL` : LM Studio 서버 주소. 컨테이너 안의 localhost 는 컨테이너 자신이므로 내 PC 는 `host.docker.internal`. 포트가 1234 가 아니면 바꾼다
- `EMBED_MODEL` : LM Studio 의 모델 이름. 셀 6 이 로드된 모델 목록을 보여 주므로 다르면 그 이름으로 바꾼다

In [ ]:
# 셀 1 · 설정
DB_URL       = "postgresql://lab:lab@db:5432/lab"       # 컨테이너끼리는 서비스 이름 db 로 접속
LMSTUDIO_URL = "http://host.docker.internal:1234/v1"    # LM Studio 서버 주소 (포트가 다르면 1234 를 바꾼다)
EMBED_MODEL  = "text-embedding-qwen3-embedding-8b"      # LM Studio 에 로드된 임베딩 모델 이름

DATA_DIR = "/cache/hf/parquet"      # 데이터셋 저장 위치 = 도커 볼륨 hfcache (컨테이너를 지워도 남는다)
SPLIT    = "validation"             # KLUE-MRC validation : 지문 5,309 · 질문 5,841

BATCH_SIZE = 16      # 한 번에 LM Studio 로 보내는 문장 수
PAUSE_SEC  = 0.0     # 배치 사이 쉬는 시간(초). PC 가 버거우면 0.5~2
MAX_EMBED  = None    # 이번 실행에서 테이블당 최대 몇 건 임베딩할지 (None = 전부)

# 질문(질의) 쪽에 붙이는 지시문. 문서 쪽에는 붙이지 않는다. Qwen3-Embedding 은 영어 지시문을 권장
QUERY_TASK = "Given a Korean question, retrieve the news or Wikipedia passage that answers the question"

### 셀 2 · 라이브러리와 DB 연결
`register_vector` 를 해 두면 numpy 배열을 pgvector 의 vector 타입으로 그대로 주고받는다.

In [ ]:
# 셀 2 · 라이브러리와 DB 연결
import hashlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
import requests
from pgvector.psycopg import register_vector

conn = psycopg.connect(DB_URL)      # 배치마다 직접 COMMIT 한다
register_vector(conn)


def one(sql, params=None):
    """SQL 결과의 첫 행 첫 값"""
    with conn.cursor() as cur:
        cur.execute(sql, params)
        row = cur.fetchone()
    return row[0] if row else None


print("DB       :", one("SELECT version()").split(",")[0])
print("pgvector :", one("SELECT extversion FROM pg_extension WHERE extname = 'vector'"))

### 셀 3 · 데이터셋 받기 (볼륨에 없을 때만)
허깅페이스가 만들어 둔 parquet 파일(약 8MB)을 받는다. 받는 도중 끊겨도 반쪽 파일이 남지 않게 `.part` 로 받고 끝나면 이름을 바꾼다.

In [ ]:
# 셀 3 · 데이터셋 받기 (볼륨에 없을 때만)
path = Path(DATA_DIR) / f"mrc-{SPLIT}.parquet"
if path.exists():
    print(f"볼륨에 있음 → 다운로드 생략 : {path} ({path.stat().st_size / 2**20:.1f} MB)")
else:
    url = f"https://huggingface.co/api/datasets/klue/klue/parquet/mrc/{SPLIT}/0.parquet"
    print("다운로드 :", url)
    path.parent.mkdir(parents=True, exist_ok=True)
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    tmp = path.with_suffix(".part")
    tmp.write_bytes(r.content)
    tmp.rename(path)
    print(f"저장 : {path} ({path.stat().st_size / 2**20:.1f} MB)")

raw = pd.read_parquet(path)
print(f"원본 {len(raw):,}행 (질문 1개 = 1행)")
raw[["title", "question", "context"]].head(3)

### 셀 4 · 지문·질문 표로 정리
원본은 질문 단위라 같은 지문이 여러 번 나온다 → 지문 내용의 md5(`doc_key`)로 중복을 지워 지문 표를 만들고, 질문 표에는 `doc_key` 로 정답 지문을 연결한다.

In [ ]:
# 셀 4 · 지문·질문 표로 정리
raw["doc_key"] = raw["context"].map(lambda s: hashlib.md5(s.encode()).hexdigest())

docs = (raw.drop_duplicates("doc_key")
           .loc[:, ["doc_key", "title", "context", "news_category", "source"]]
           .rename(columns={"context": "content", "news_category": "category"})
           .reset_index(drop=True))
docs["category"] = docs["category"].replace({"": None})        # 위키 지문은 분야 없음

qs = raw.loc[:, ["guid", "doc_key", "question", "question_type", "is_impossible"]].copy()
qs["answer"] = raw["answers"].map(lambda a: a["text"][0] if len(a["text"]) else None)   # 답 없는 질문은 None

print(f"지문 {len(docs):,}개 / 질문 {len(qs):,}개")
docs.head(3)

### 셀 5 · 테이블 비우고 텍스트 적재
테이블(docs · questions)은 DB 가 처음 뜰 때 `db/init` 이 이미 만들어 둔다. 여기서는 비우고 채우기만 한다.
- `TRUNCATE … RESTART IDENTITY CASCADE` : 한 번에 비우고 id 를 1부터 다시 → 몇 번을 적재해도 같은 지문은 같은 id. docs 를 참조하는 테이블(4-2 의 doc_chunks)도 함께 비운다
- `INSERT … SELECT FROM unnest(배열, 배열, …)` : 열마다 파이썬 리스트 하나를 배열로 넘겨 **INSERT 한 번**에 전부 넣는다. `tsv` 는 생성 컬럼이라 이때 형태소 분석까지 끝난다
- 질문의 `doc_id`(정답 지문)는 SQL 안에서 `doc_key` 로 docs 와 조인해 채운다. `WITH ORDINALITY … ORDER BY` 로 원본 순서대로 id 를 매긴다
- docs 에는 BM25 트리거가 걸려 있다 : TRUNCATE 트리거가 역색인을 비우고, INSERT 트리거가 새 지문으로 역색인(bm25_tf · bm25_df)을 채운다 (그래서 약 50초 걸린다)
- 이 셀이 끝나면 키워드 검색 실습은 다른 노트북에서 바로 시작해도 된다

In [ ]:
# 셀 5 · 테이블 비우고 텍스트 적재
t0 = time.time()
with conn.cursor() as cur:
    cur.execute("DROP INDEX IF EXISTS docs_emb128_hnsw")        # 임베딩을 채우는 동안에는 인덱스를 두지 않는다
    cur.execute("TRUNCATE docs, questions RESTART IDENTITY CASCADE")

    # 지문 : 열마다 배열 하나씩 넘겨 INSERT 한 번
    cur.execute("""
        INSERT INTO docs (doc_key, title, content, category, source)
        SELECT * FROM unnest(%s::text[], %s::text[], %s::text[], %s::text[], %s::text[])""",
        [docs[c].tolist() for c in ["doc_key", "title", "content", "category", "source"]])

    # 질문 : doc_key 로 docs 와 조인해 doc_id(정답 지문)를 채운다. 원본 순서대로 id 가 매겨지게 ORDER BY
    cur.execute("""
        INSERT INTO questions (guid, doc_id, question, answer, question_type, is_impossible)
        SELECT q.guid, d.id, q.question, q.answer, q.question_type, q.is_impossible
        FROM unnest(%s::text[], %s::text[], %s::text[], %s::text[], %s::int[], %s::bool[])
             WITH ORDINALITY AS q(guid, doc_key, question, answer, question_type, is_impossible, n)
        JOIN docs d USING (doc_key)
        ORDER BY q.n""",
        [qs[c].tolist() for c in ["guid", "doc_key", "question", "answer", "question_type", "is_impossible"]])

    cur.execute("ANALYZE docs")
    cur.execute("ANALYZE questions")
conn.commit()
print(f"텍스트 적재 {time.time() - t0:.1f}s : docs {one('SELECT count(*) FROM docs'):,} · questions {one('SELECT count(*) FROM questions'):,}")
print("✔ 텍스트 적재 완료")

### 셀 6 · 임베딩 서버 확인
LM Studio 에 로드된 모델 목록을 보여 주고, 셀 1 의 `EMBED_MODEL` 로 시험 문장 하나를 임베딩해 4096차원인지 본다.
`embed()` 는 여러 문장을 요청 한 번으로 보내고, 실패하면 5 · 10 · 20초 쉬고 다시 시도한다.
**멈춘 뒤 이어서 할 때는 이 셀부터** Run Selected Cell and All Below.

In [ ]:
# 셀 6 · 임베딩 서버 확인
models = [m["id"] for m in requests.get(f"{LMSTUDIO_URL}/models", timeout=10).json()["data"]]
print("LM Studio 에 로드된 모델 :", models)
assert EMBED_MODEL in models, f"셀 1 의 EMBED_MODEL 을 위 목록의 이름으로 바꾼다 (지금 값: {EMBED_MODEL})"


def embed(texts, retries=4):
    """POST /v1/embeddings 로 여러 문장을 한 번에 임베딩"""
    for attempt in range(retries):
        try:
            r = requests.post(f"{LMSTUDIO_URL}/embeddings", json={"model": EMBED_MODEL, "input": texts}, timeout=600)
            r.raise_for_status()
            data = sorted(r.json()["data"], key=lambda d: d.get("index", 0))
            return [np.asarray(d["embedding"], dtype=np.float32) for d in data]
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = 5 * 2 ** attempt
            print(f"  ! 요청 실패({type(e).__name__}) → {wait}s 뒤 재시도")
            time.sleep(wait)


v = embed(["반응기 온도가 계속 오르는 원인"])[0]
print(f"{v.size}차원 · 길이 {np.linalg.norm(v):.4f} · 앞 128차원만 잘랐을 때 길이 {np.linalg.norm(v[:128]):.4f}")
assert v.size == 4096, "4096차원이 아니다. 모델을 확인할 것"

### 셀 7 · 배치 임베딩 함수
1. `embedding IS NULL` 인 행을 id 순으로 `BATCH_SIZE` 개 읽는다
2. 그 텍스트를 요청 한 번으로 임베딩한다 (`prefix` 가 있으면 앞에 붙인다)
3. UPDATE 하고 COMMIT → `emb128` 은 생성 컬럼이라 DB 가 자동으로 채운다
4. `PAUSE_SEC` 쉬고 1로. 20배치(320건)마다 진행 상황을 출력

비어 있는 행만 고르므로, 멈췄다 다시 돌리면 남은 행부터 이어진다.

In [ ]:
# 셀 7 · 배치 임베딩 함수
def embed_pending(table, text_sql, prefix=""):
    remaining = one(f"SELECT count(*) FROM {table} WHERE embedding IS NULL")
    todo = remaining if MAX_EMBED is None else min(remaining, MAX_EMBED)
    print(f"[{table}] 비어 있는 행 {remaining:,} → 이번에 {todo:,}건 (배치 {BATCH_SIZE}, 쉬는 시간 {PAUSE_SEC}s)")
    done, nb, t0 = 0, 0, time.time()
    while done < todo:
        with conn.cursor() as cur:                                         # 1) 빈 행 읽기
            cur.execute(f"SELECT id, {text_sql} FROM {table} WHERE embedding IS NULL ORDER BY id LIMIT %s",
                        (min(BATCH_SIZE, todo - done),))
            rows = cur.fetchall()
        if not rows:
            break
        vecs = embed([prefix + text for _, text in rows])                  # 2) 임베딩
        with conn.cursor() as cur:                                         # 3) 저장
            cur.executemany(f"UPDATE {table} SET embedding = %s WHERE id = %s",
                            [(vec, id_) for (id_, _), vec in zip(rows, vecs)])
        conn.commit()
        done += len(rows)
        nb += 1
        if nb % 20 == 0 or done >= todo:
            el = time.time() - t0
            rate = done / el
            print(f"  {done:6,}/{todo:,}  {rate:5.1f}건/s  경과 {el / 60:5.1f}분  남은 시간 약 {(todo - done) / rate / 60:5.1f}분")
        if PAUSE_SEC:                                                      # 4) 쉬기
            time.sleep(PAUSE_SEC)
    print(f"[{table}] {done:,}건 완료. 남은 행 {one(f'SELECT count(*) FROM {table} WHERE embedding IS NULL'):,}")

### 셀 8 · 지문 임베딩
문서 쪽은 **제목 + 줄바꿈 + 본문**, 지시문 없음. 가장 오래 걸리는 셀이다.

In [ ]:
# 셀 8 · 지문 임베딩
embed_pending("docs", "coalesce(title, '') || E'\\n' || content")

### 셀 9 · 질문 임베딩
질문(질의) 쪽은 모델카드 형식대로 지시문을 붙인다 : `Instruct: {QUERY_TASK}` + 줄바꿈 + `Query:{질문}`

In [ ]:
# 셀 9 · 질문 임베딩
embed_pending("questions", "question", prefix=f"Instruct: {QUERY_TASK}\nQuery:")

### 셀 10 · 인덱스 만들기와 확인
지문 임베딩이 다 찼을 때만 `emb128` 에 HNSW(코사인) 인덱스를 만든다. 원본 4096 에는 만들 수 없다(pgvector HNSW 상한 2,000차원).
확인 표 : 원본 길이는 1, 앞 128차원만 자르면 약 0.2, `emb128` 은 다시 1.

In [ ]:
# 셀 10 · 인덱스 만들기와 확인
if one("SELECT count(*) FROM docs WHERE embedding IS NULL") == 0:
    t0 = time.time()
    with conn.cursor() as cur:
        cur.execute("CREATE INDEX IF NOT EXISTS docs_emb128_hnsw ON docs USING hnsw (emb128 vector_cosine_ops)")
        cur.execute("ANALYZE docs")
    conn.commit()
    print(f"docs_emb128_hnsw 생성 {time.time() - t0:.1f}s")
else:
    print("지문 임베딩이 아직 남아 있다 → 셀 6 부터 다시 실행한 뒤 인덱스를 만든다")

with conn.cursor() as cur:
    cur.execute("""
      SELECT t, count(*) AS 행, count(embedding) AS 임베딩,
             round(avg(vector_norm(embedding))::numeric, 4)                     AS 원본_길이,
             round(avg(vector_norm(subvector(embedding, 1, 128)))::numeric, 4)  AS 앞128_길이,
             round(avg(vector_norm(emb128))::numeric, 4)                        AS emb128_길이
      FROM (SELECT 'docs' t, embedding, emb128 FROM docs
            UNION ALL SELECT 'questions', embedding, emb128 FROM questions) x
      GROUP BY t ORDER BY t""")
    print(pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description]).to_string(index=False))

### 셀 11 · 맛보기 검색
같은 질문을 원본 4096 전수 비교와 emb128(HNSW)로 검색해 상위 5개를 나란히 본다. ★ = 정답 지문.
`QID` 를 바꿔 다른 질문도 해 본다.

In [ ]:
# 셀 11 · 맛보기 검색
QID = 1
with conn.cursor() as cur:
    cur.execute("SELECT question, doc_id FROM questions WHERE id = %s", (QID,))
    question, gold = cur.fetchone()
    print("질문 :", question, f"(정답 지문 id {gold})\n")
    for name, sql in [
        ("원본 4096 전수 비교", "SELECT d.id, d.title, 1 - (d.embedding <=> q.embedding) FROM docs d, questions q "
                                "WHERE q.id = %s ORDER BY d.embedding <=> q.embedding LIMIT 5"),
        ("emb128 (HNSW)",       "SELECT d.id, d.title, 1 - (d.emb128 <=> q.emb128) FROM docs d, questions q "
                                "WHERE q.id = %s ORDER BY d.emb128 <=> q.emb128 LIMIT 5"),
    ]:
        cur.execute(sql, (QID,))
        print(f"[{name}]")
        for id_, title, sim in cur.fetchall():
            print(f"  {'★' if id_ == gold else ' '} {sim:.4f}  {id_:5d}  {title}")
        print()